# Python 结构化模式匹配（match 语句）教学级详解

> **学习目标**
> 1. 掌握 `match / case` 的完整语法与 10 类模式（字面量、捕获、通配符、序列、映射、类、OR、守卫、值、组合）
> 2. 能用模式匹配解构嵌套的 JSON / 字典 / 元组数据，替代复杂 if-elif 链
> 3. 能识别并规避 12 类常见陷阱（捕获吞噬、常量误当比较、`__match_args__` 缺失等）
> 4. 能在 LLM API 网关等真实工程场景中组合运用多种模式

> **一句话本质总结**
> 模式匹配不是简单的 switch-case，而是 **「检查结构 + 解构数据 + 绑定变量」三合一** 的工具；它最适合处理**嵌套、异构、结构化**的数据。

> **运行环境要求**：Python **3.10+**（本文所有代码在 Python 3.13 上实测执行，输出全部嵌入）


In [1]:
# 0-1 环境自检：match 语句需要 Python 3.10 及以上
import sys                                    # ① 导入 sys 模块，用于读取解释器版本
print("Python 版本:", sys.version)             # ② 打印完整版本信息
assert sys.version_info >= (3, 10), "需要 Python 3.10+"   # ③ 断言版本满足要求，否则报错
print("版本检查通过，可以学习 match 语句")       # ④ 验证通过的提示

Python 版本: 3.13.12 (main, Feb  4 2026, 00:05:32) [Clang 21.1.4 ]
版本检查通过，可以学习 match 语句


## 一、前置知识：match 依赖哪些语言基础？

**本章解决什么问题**：在正式学习 match 之前，先把它的三块地基打牢——**序列解包**、**isinstance 类型检查**、**字典键值访问**。match 的序列/类/映射模式本质上是这三者的语法糖升级版。

| 前置特性 | 在 match 中的对应物 | 关键点 |
|---|---|---|
| `a, b = (1, 2)` 位置解包 | 序列模式 `case (a, b):` | 位置对应、可嵌套 |
| `first, *rest = [1,2,3]` 星号解包 | 变长序列 `case [first, *rest]:` | `*` 收集剩余元素 |
| `isinstance(x, int)` | 类模式 `case int():` | 检查类型而非值 |
| `d["name"]` / `d.get("name")` | 映射模式 `case {"name": n}:` | 键存在性检查 + 值绑定 |
| `getattr(obj, "x")` | 关键字类模式 `case Point(x=0):` | 按属性名匹配 |

下面逐一用可运行代码验证这些基础。

In [2]:
# 1-1 前置知识①：位置解包与星号解包
# ① 基本位置解包：左边变量个数必须与右边序列长度一致
a, b = (1, 2)
print("位置解包:", a, b)                # 输出: 1 2

# ② 星号解包：*rest 收集剩余所有元素，结果是列表
first, *rest = [1, 2, 3, 4]
print("星号解包:", first, rest)          # 输出: 1 [2, 3, 4]

# ③ 首尾固定 + 中间收集
first, *middle, last = [1, 2, 3, 4]
print("首尾解包:", first, middle, last)  # 输出: 1 [2, 3] 4

# ④ 双星号解包发生在「字典字面量 / 函数调用」里，不能用于赋值目标
#    —— 这正是 match 映射模式 case {"a": a, **rest}: 的价值：模式里才允许 **rest！
merged = {"a": 1, **{"b": 2, "c": 3}}       # 字典字面量中的 ** 合并
print("字典字面量 ** 合并:", merged)         # 输出: {'a': 1, 'b': 2, 'c': 3}

def kw_info(a, **rest):                     # 函数签名中的 **rest 收集剩余关键字参数
    return a, rest                          # rest 是 dict
print("函数调用 ** 收集:", kw_info(a=1, b=2, c=3))   # 输出: (1, {'b': 2, 'c': 3})

位置解包: 1 2
星号解包: 1 [2, 3, 4]
首尾解包: 1 [2, 3] 4
字典字面量 ** 合并: {'a': 1, 'b': 2, 'c': 3}
函数调用 ** 收集: (1, {'b': 2, 'c': 3})


In [3]:
# 1-2 前置知识②：isinstance 与属性访问（类模式的地基）
class Point:                              # ① 定义一个简单的二维点类
    def __init__(self, x, y):             # ② 构造函数接收 x、y
        self.x = x                        # ③ 实例属性 x
        self.y = y                        # ④ 实例属性 y

p = Point(1, 2)                           # ⑤ 创建实例
print("isinstance 检查:", isinstance(p, Point))     # ⑥ True：p 是 Point 的实例
print("属性访问:", getattr(p, "x"))                  # ⑦ 等价于 p.x，输出 1
print("hasattr 检查:", hasattr(p, "x"))              # ⑧ True：属性存在

# ⑨ match 的类模式 Point(x=0) 等价于：
#    isinstance(subject, Point) and subject.x == 0
print("等价逻辑:", isinstance(p, Point) and getattr(p, "x") == 1)  # ⑩ True

isinstance 检查: True
属性访问: 1
hasattr 检查: True
等价逻辑: True


In [4]:
# 1-3 前置知识③：字典的键存在性检查（映射模式的地基）
d = {"name": "Alice", "age": 30}          # ① 一个典型的 API 响应字典

# ② 传统写法：逐键检查 + 取值，嵌套深了非常啰嗦
if "name" in d and "age" in d:
    print("传统写法:", d["name"], d["age"])

# ③ .get() 的局限：无法区分「键不存在」和「值是 None」
print("get 默认值:", d.get("city", "未知"))  # 键不存在时返回默认值: 未知

# ④ match 映射模式 case {"name": n, "age": a}: 的语义等价于：
#    isinstance(d, dict) and "name" in d and "age" in d
#    并且把 d["name"] 绑定给 n、d["age"] 绑定给 a
print("等价逻辑:", isinstance(d, dict) and "name" in d and "age" in d)  # ⑤ True

传统写法: Alice 30
get 默认值: 未知
等价逻辑: True


## 二、正文：从基本语法到组合模式

**本章解决什么问题**：按知识脉络逐个击破 10 类模式。每个知识点 = 原理讲解（Markdown）+ 完整可运行示例（Code，输出已嵌入）。

### 2.1 基本语法

`match` 后跟要匹配的对象（subject），`case` 后跟模式，`_` 是通配符。三条核心规则：

- **从上到下依次匹配**，第一个匹配成功的分支执行
- **case 不会“穿透”**（fall-through），匹配一个就结束，不需要 break
- 如果没有匹配且没有 `_`，**什么都不做**（不报错）

In [5]:
# 2-1 基本语法：HTTP 状态码分发
def http_status(code):                    # ① 定义函数：把状态码翻译成语义
    match code:                           # ② match 后跟被匹配对象（subject）
        case 200:                         # ③ 字面量模式：code == 200 时命中
            return "OK"                   # ④ 命中后直接 return，不会穿透到下一个 case
        case 404:                         # ⑤ 同理：404
            return "Not Found"
        case 500:
            return "Server Error"
        case _:                           # ⑥ 通配符：兜底，匹配任何值（必须放最后）
            return "Unknown"

print(http_status(200))   # ⑦ OK
print(http_status(404))   # ⑧ Not Found
print(http_status(999))   # ⑨ Unknown（落入通配符）

# ⑩ 验证「无匹配且无 _ 时静默通过」：这里故意去掉通配符
def silent(x):
    match x:
        case 1:
            return "one"
    return "没有匹配也不报错"              # ⑪ x=2 时两个分支都不命中，直接走到这里
print(silent(2))          # ⑫ 输出: 没有匹配也不报错

OK
Not Found
Unknown
没有匹配也不报错


### 2.2 字面量模式

匹配**具体的值**。支持的字面量：

| 类型 | 示例 |
|---|---|
| 整数 / 浮点 / 复数 | `0`、`1.5`、`1+2j` |
| 字符串 | `"hello"` |
| 布尔 | `True`、`False` |
| None | `None` |

⚠️ 注意：`True`、`False`、`None` 是**关键字字面量**，不是变量捕获——这是它们和普通名字 `x` 的本质区别（普通名字一律是捕获模式）。

In [6]:
# 2-2 字面量模式：注意 1 与 True 的微妙区别
def describe(x):
    match x:
        case 0:          # ① 整数字面量
            return "零"
        case 1:          # ② 整数字面量（在 True 之前，先被 1 命中吗？看下方验证）
            return "一"
        case "hello":    # ③ 字符串字面量
            return "问候"
        case True:       # ④ 布尔字面量（关键字，不是捕获）
            return "真"
        case None:       # ⑤ None 字面量（关键字，不是捕获）
            return "空"
        case _:          # ⑥ 兜底
            return "其他"

print(describe(0))       # ⑦ 零
print(describe(1))       # ⑧ 一 —— int(1) 先命中 case 1（顺序在前的优先）
print(describe(True))    # ⑨ 一？不——验证见下：True 是 bool 类型
print(describe("hello")) # ⑩ 问候
print(describe(None))    # ⑪ 空
print(describe(3.14))    # ⑫ 其他

# ⑬ 关键验证：True 会先被 case 1 命中还是 case True？
def bool_order(x):
    match x:
        case 1:          # ⑭ 字面量 1 在前
            return "命中 case 1"
        case True:       # ⑮ 字面量 True 在后
            return "命中 case True"
print("bool_order(True):", bool_order(True))   # ⑯ 输出: 命中 case 1（1 == True 且 1 在前）
print("bool_order(1):", bool_order(1))         # ⑰ 输出: 命中 case 1

零
一
一
问候
空
其他
bool_order(True): 命中 case 1
bool_order(1): 命中 case 1


### 2.3 捕获模式

用**变量名**捕获值：`case x:` 会把 subject 绑定到 `x`，并且**匹配任何值**。

三个要点：
1. 捕获模式匹配一切 → 通常放在**最后**当兜底（但更推荐用 `_`）
2. `_` 是特殊捕获模式，表示“不关心这个值”，**不会绑定变量**
3. `case MAX:`（大写常量）也是**捕获**，不是比较！这是最高频的坑（详见 2.3.2）

In [7]:
# 2-3 捕获模式：变量绑定
match 42:                # ① subject 是字面量 42
    case x:              # ② x 是捕获模式：匹配任何值并把 42 绑定给 x
        print("x =", x)  # ③ 输出: x = 42

def f(x):
    match x:
        case 0:          # ④ 字面量模式在前，优先匹配
            return "零"
        case n:          # ⑤ 捕获模式兜底：捕获所有非 0 的值
            return f"其他: {n}"
print(f(0))              # ⑥ 零
print(f(99))             # ⑦ 其他: 99

x = 42
零
其他: 99


In [8]:
# 2-3.1 通配符 _：匹配但不绑定
_ = "哨兵值"                              # ① 故意先给 _ 赋一个哨兵值
match (1, 2):                             # ② 匹配二元组
    case (_, y):                          # ③ 第一个位置用 _：不关心，也不绑定
        print("y =", y)                   # ④ 输出: y = 2
print("_ 匹配后仍是:", _)                  # ⑤ 输出: 哨兵值 —— 证明 _ 没有被绑定成 1

y = 2
_ 匹配后仍是: 哨兵值


#### 2.3.2 捕获与常量的陷阱（高频！）

`case MAX:` 里的 `MAX` 是**捕获模式**（绑定），不是「x == MAX 的比较」。因为模式中的**裸名字**（不带点号）永远被解析为捕获。要比较常量，用字面量 `case 100:` 或守卫 `case n if n == MAX:`。

In [9]:
# 2-3.2 验证：case MAX 是捕获，不是比较
MAX = 100                                   # ① 定义一个模块级常量

def check(x):
    match x:
        case MAX:                           # ② 注意：这不是比较！MAX 是裸名字 → 捕获模式
            return f"进入分支, x={x}, 局部MAX={MAX}"   # ③ MAX 被重新绑定为 x 的值
    return "不会走到这里"                     # ④ 捕获匹配一切，永远命中上一分支

print(check(7))       # ⑤ 输出: 进入分支, x=7, 局部MAX=7 —— 7 != 100 却命中了！
print(check(100))     # ⑥ 同样命中

# ⑦ 正确写法一：字面量
def check_v1(x):
    match x:
        case 100:
            return "等于 100"
        case _:
            return "不等于 100"

# ⑧ 正确写法二：守卫 + 捕获
def check_v2(x):
    match x:
        case n if n == MAX:    # ⑨ 先捕获到 n，再用守卫比较
            return "等于 MAX"
        case _:
            return "不等于 MAX"

print(check_v1(100), "|", check_v1(7))       # ⑩ 等于 100 | 不等于 100
print(check_v2(100), "|", check_v2(7))       # ⑪ 等于 MAX | 不等于 MAX

进入分支, x=7, 局部MAX=7
进入分支, x=100, 局部MAX=100
等于 100 | 不等于 100
等于 MAX | 不等于 MAX


### 2.4 序列模式

匹配**列表、元组**等序列的**结构**。四条规则：

1. 匹配的是**结构**，不是类型：元组可以匹配列表模式，反之亦然
2. `*rest` 收集剩余元素，`*` 在一个模式中**只能出现一次**
3. 支持任意**嵌套**解构
4. ⚠️ **`str` / `bytes` / `bytearray` 被显式排除**，不能被序列模式解构（见 2.4.3 修正说明）

> **【修正说明】** 源文档称“字符串也可以按字符解构”，这是**错误的**。PEP 634 明确规定序列模式不匹配 `str`、`bytes`、`bytearray`。下方代码用实际输出验证。

In [10]:
# 2-4 序列模式基础：完全匹配与解构绑定
match [1, 2, 3]:                 # ① subject 是列表
    case [1, 2, 3]:              # ② 完全匹配：长度和每个位置的值都对上
        print("完全匹配")         # ③ 命中

match (1, 2):                    # ④ subject 是元组
    case (a, b):                 # ⑤ 解构：a=1, b=2（圆括号可省略：case a, b:）
        print("解构绑定:", a, b)  # ⑥ 输出: 1 2

match [1, 2]:                    # ⑦ 列表 subject
    case (a, b):                 # ⑧ 元组模式照样匹配 —— 结构匹配，不区分 list/tuple
        print("跨类型匹配:", a, b) # ⑨ 输出: 1 2

完全匹配
解构绑定: 1 2
跨类型匹配: 1 2


In [11]:
# 2-4.1 变长序列：*rest 与 *middle
match [1, 2, 3, 4]:                      # ① 4 个元素的列表
    case [first, *rest]:                 # ② first 吃第一个，*rest 收集剩余
        print("first, rest =", first, rest)      # ③ 输出: 1 [2, 3, 4]

match [1, 2, 3, 4]:
    case [first, *middle, last]:         # ④ 首尾固定，中间收集
        print("first, middle, last =", first, middle, last)   # ⑤ 1 [2, 3] 4

# ⑥ 验证「* 只能出现一次」：写两个 * 是语法错误
try:
    compile("match x:\n    case [a, *b, *c]:\n        pass", "<demo>", "exec")
    print("两个 * 竟然编译通过（不应该发生）")
except SyntaxError as e:                 # ⑦ 捕获语法错误，证明结论
    print("两个 * -> SyntaxError:", e.msg)

# ⑧ *rest 也可以匹配空序列
match [1]:
    case [first, *rest]:                 # ⑨ rest 收集到空列表 []
        print("空 rest:", first, rest)    # ⑩ 输出: 1 []

first, rest = 1 [2, 3, 4]
first, middle, last = 1 [2, 3] 4
两个 * -> SyntaxError: multiple starred names in sequence pattern
空 rest: 1 []


In [12]:
# 2-4.2 嵌套序列
match [1, [2, 3]]:                    # ① 外层列表，第二个元素又是列表
    case [a, [b, c]]:                 # ② 嵌套解构：外层两个位置，内层两个位置
        print(a, b, c)                # ③ 输出: 1 2 3

# ④ 更深的三层嵌套
match [1, [2, [3, 4]]]:
    case [a, [b, [c, d]]]:            # ⑤ 逐层剥开
        print("三层嵌套:", a, b, c, d)  # ⑥ 输出: 1 2 3 4

1 2 3
三层嵌套: 1 2 3 4


#### 2.4.3 字符串与序列模式 ——【修正说明】

**源文档原文**：“字符串也可以按字符解构，但一般用字面量更清晰。”

**修正**：这句话的前半句是错的。`str`、`bytes`、`bytearray` **不会**被序列模式匹配/解构——这是 PEP 634 的刻意设计（防止 `case "hello":` 被误写成 `case [h, e, l, l, o]:` 造成静默不匹配）。要处理字符串内容请用**字面量模式**或**守卫 + 普通字符串方法**。

In [13]:
# 2-4.3 验证修正：str/bytes/bytearray 不被序列模式匹配
for subj in ["ab", b"ab", bytearray(b"ab")]:     # ① 三种被排除的类型
    result = "未匹配"                              # ② 默认结果
    match subj:                                   # ③ 尝试用两元素序列模式匹配
        case [a, b]:
            result = f"匹配成功 {a!r} {b!r}"
    print(f"{type(subj).__name__:<10} -> {result}")   # ④ 全部输出「未匹配」

# ⑤ 正确处理字符串的方式一：字面量模式
match "hello":
    case "hello":                       # ⑥ 整体字面量比较
        print("字面量匹配成功")

# ⑥ 正确处理字符串的方式二：守卫 + 字符串方法
def classify(s):
    match s:
        case t if isinstance(t, str) and t.startswith("/"):   # ⑦ 守卫里做字符串判断
            return "斜杠命令"
        case _:
            return "普通文本"
print(classify("/deploy"))    # ⑧ 斜杠命令
print(classify("你好"))       # ⑨ 普通文本

str        -> 未匹配
bytes      -> 未匹配
bytearray  -> 未匹配
字面量匹配成功
斜杠命令
普通文本


In [14]:
# 2-4.4 序列匹配规则：结构优先、类型无关
match (1, 2, 3):                 # ① 元组 subject
    case [1, 2, 3]:              # ② 列表模式 —— 依然命中
        print("元组匹配列表模式: 命中")   # ③ 证明：匹配的是结构不是类型

# ④ 但长度必须完全一致（除非用 *）
match (1, 2):
    case [1, 2, 3]:              # ⑤ 长度不符
        print("不会命中")
    case _:                      # ⑥ 落入通配符
        print("长度不符 -> 未命中")

元组匹配列表模式: 命中
长度不符 -> 未命中


### 2.5 映射模式

匹配**字典结构**。四条规则：

1. **部分匹配**：只要求包含指定的键，不要求键完全一致（与序列模式的“长度一致”相反！）
2. 支持任意**嵌套**
3. `**rest` 收集剩余键值对
4. 键必须是**字面量或值模式（点号访问）**，裸变量名做键是**语法错误**

In [15]:
# 2-5 映射模式基础 + 部分匹配
match {"name": "Alice", "age": 30}:        # ① 完整字典
    case {"name": name, "age": age}:       # ② 解构：键做匹配，值做绑定
        print("基础解构:", name, age)       # ③ Alice 30

# ④ 部分匹配：字典多出的键不影响命中
match {"name": "Alice", "age": 30, "city": "Beijing"}:
    case {"name": name}:                   # ⑤ 只检查 name 键
        print("部分匹配:", name)             # ⑥ Alice —— 多出的 age/city 被忽略

# ⑦ 嵌套映射
data = {
    "user": {"name": "Alice", "age": 30},  # ⑧ 嵌套的 user 字典
    "active": True,
}
match data:
    case {"user": {"name": name, "age": age}, "active": True}:   # ⑨ 双层嵌套 + 字面量 True
        print("嵌套匹配:", name, age)       # ⑩ Alice 30

基础解构: Alice 30
部分匹配: Alice
嵌套匹配: Alice 30


In [16]:
# 2-5.3 剩余键：**rest 收集
match {"a": 1, "b": 2, "c": 3}:           # ① 三个键的字典
    case {"a": a, **rest}:                # ② a 绑定键 a 的值，**rest 收集其余键值对
        print("a =", a)                   # ③ 1
        print("rest =", rest)             # ④ {'b': 2, 'c': 3}

# ⑤ 验证「映射模式的键不能是裸变量名」：语法错误
try:
    compile("match d:\n    case {key: value}:\n        pass", "<demo>", "exec")
    print("裸名字做键竟然编译通过（不应该发生）")
except SyntaxError as e:                  # ⑥ 捕获语法错误
    print("裸变量做键 -> SyntaxError:", e.msg)

# ⑦ 正确：键必须是字面量（或下面 2.9 的值模式如 Color.RED）
match {"name": "Alice"}:
    case {"name": n}:                     # ⑧ "name" 是字符串字面量键
        print("字面量键 OK:", n)

a = 1
rest = {'b': 2, 'c': 3}
裸变量做键 -> SyntaxError: invalid syntax
字面量键 OK: Alice


### 2.6 类模式

匹配对象的**类型和属性**。两步语义：① `isinstance(subject, 类)` 检查；② 按位置或关键字匹配属性。

| 写法 | 前提条件 | 说明 |
|---|---|---|
| `case Point(x=0, y=0):` | 无特殊要求 | 关键字模式，按属性名匹配 |
| `case Point(0, 0):` | 类定义了 `__match_args__` | 位置模式，顺序由 `__match_args__` 决定 |
| `case Point():` | 无特殊要求 | 只做 isinstance 检查 |
| `case int():` / `case float():` | — | 内置类同理，可加参数如 `case str()` |

In [17]:
# 2-6 类模式：关键字属性匹配
class Point:
    def __init__(self, x, y):     # ① 注意：这个类没有定义 __match_args__
        self.x = x                # ②
        self.y = y                # ③

def where(p):
    match p:                              # ④ subject 是 Point 实例
        case Point(x=0, y=0):             # ⑤ 关键字模式：x 属性 == 0 且 y 属性 == 0
            return "原点"
        case Point(x=0, y=y):             # ⑥ x 属性 == 0，y 属性绑定给变量 y
            return f"y 轴上, y={y}"
        case Point(x=x, y=0):             # ⑦ x 绑定，y == 0
            return f"x 轴上, x={x}"
        case Point(x=x, y=y):             # ⑧ 兜底：两个属性都绑定
            return f"普通点 ({x}, {y})"

print(where(Point(0, 0)))    # ⑨ 原点
print(where(Point(0, 5)))    # ⑩ y 轴上, y=5
print(where(Point(3, 0)))    # ⑪ x 轴上, x=3
print(where(Point(3, 4)))    # ⑫ 普通点 (3, 4)

原点
y 轴上, y=5
x 轴上, x=3
普通点 (3, 4)


In [18]:
# 2-6.1 位置类模式：__match_args__ 定义位置顺序
class Point:
    __match_args__ = ("x", "y")    # ① 声明位置模式的参数顺序：第 1 个对应 x，第 2 个对应 y

    def __init__(self, x, y):
        self.x = x
        self.y = y

match Point(1, 2):
    case Point(0, 0):              # ② 位置字面量：等价于 Point(x=0, y=0)
        print("原点")
    case Point(x, y):              # ③ 位置捕获：x=1, y=2
        print(f"位置捕获: {x}, {y}")   # ④ 输出: 位置捕获: 1, 2

位置捕获: 1, 2


In [19]:
# 2-6.2 内置类模式：类型检查的最简写法
match 3.5:                         # ① 浮点数
    case float():                  # ② 等价于 isinstance(3.5, float)
        print("浮点数")             # ③ 命中

match "hello":
    case str():                    # ④ 字符串
        print("字符串")

match [1, 2]:
    case list():                   # ⑤ 列表
        print("列表")

# ⑥ 常见搭配：类型模式 + 捕获（as）+ 守卫
def num_info(v):
    match v:
        case int() as n if n > 0:  # ⑦ 是 int 且为正，n 绑定原值
            return f"正整数 {n}"
        case float() as f if f < 0:
            return f"负浮点 {f}"
        case _:
            return "其他"
print(num_info(42))       # ⑧ 正整数 42
print(num_info(-1.5))     # ⑨ 负浮点 -1.5

浮点数
字符串
列表
正整数 42
负浮点 -1.5


In [20]:
# 2-6.3 嵌套类模式
class Point:
    __match_args__ = ("x", "y")           # ① 点：位置参数 x, y
    def __init__(self, x, y):
        self.x, self.y = x, y

class Circle:
    __match_args__ = ("center", "radius")  # ② 圆：位置参数 center, radius
    def __init__(self, center, radius):
        self.center, self.radius = center, radius

match Circle(Point(0, 0), 5):              # ③ 圆心在原点、半径 5 的圆
    case Circle(Point(0, 0), r):           # ④ 嵌套：center 又用 Point 位置模式匹配
        print(f"圆心在原点的圆, 半径 {r}")  # ⑤ 命中，r=5
    case Circle(center, r):
        print(f"圆心在 ({center.x}, {center.y}) 的圆, 半径 {r}")

match Circle(Point(1, 2), 3):              # ⑥ 圆心不在原点
    case Circle(Point(0, 0), r):
        print("不会命中")
    case Circle(Point(x, y), r):           # ⑦ 嵌套捕获圆心坐标
        print(f"圆心在 ({x}, {y}) 的圆, 半径 {r}")   # ⑧ 圆心在 (1, 2) 的圆, 半径 3

圆心在原点的圆, 半径 5
圆心在 (1, 2) 的圆, 半径 3


In [21]:
# 2-6.4 类模式与继承：子类匹配父类模式，反之不行
class Animal:        # ① 父类
    pass

class Dog(Animal):   # ② 子类
    pass

def kind_of(o):
    match o:
        case Dog():          # ③ 子类模式在前
            return "狗"
        case Animal():       # ④ 父类模式在后
            return "动物"

print(kind_of(Dog()))      # ⑤ 狗 —— Dog 实例先命中 Dog 模式
print(kind_of(Animal()))   # ⑥ 动物 —— Animal 实例不是 Dog，落入父类模式

# ⑦ 反向验证：父类实例不匹配子类模式
match Animal():
    case Dog():
        print("不会命中")
    case Animal():
        print("父类实例只匹配父类模式")   # ⑧ 输出这行

狗
动物
父类实例只匹配父类模式


### 2.7 OR 模式

用 `|` 把多个模式连接成一个“或”模式，任意一个子模式命中即命中。两条硬规则：

1. **所有分支必须绑定相同的变量集合**，否则语法错误
2. OR 模式可以和守卫组合：`case 1 | 2 | 3 if x > 0:`（守卫作用于整个 OR 模式）

In [22]:
# 2-7 OR 模式基础
def classify(x):
    match x:
        case 1 | 2 | 3:            # ① 三个字面量的或
            return "小数字"
        case "a" | "b":            # ② 字符串的或
            return "字母"
        case _:
            return "其他"

print(classify(2))    # ③ 小数字
print(classify("b"))  # ④ 字母
print(classify(9))    # ⑤ 其他

# ⑥ OR 模式中的捕获：所有分支绑定相同变量（合法）
def get_y(pair):
    match pair:
        case [1, y] | [y, 1]:      # ⑦ 两个分支都只绑定 y —— 合法
            return y
        case _:
            return None
print(get_y([1, 99]))    # ⑧ 99（第一个分支：1 是字面量，y 捕获 99）
print(get_y([99, 1]))    # ⑨ 99（第二个分支：y 捕获 99）
print(get_y([5, 5]))     # ⑩ None

# ⑪ 验证「变量不一致是语法错误」
try:
    compile("match x:\n    case [1, a] | [b, 2]:\n        pass", "<demo>", "exec")
    print("竟然编译通过（不应该发生）")
except SyntaxError as e:
    print("OR 变量不一致 -> SyntaxError:", e.msg)   # ⑫ alternative patterns bind different names

# ⑬ OR + 守卫
def small_positive(x):
    match x:
        case 1 | 2 | 3 if x > 0:   # ⑭ 先模式匹配（或），再守卫过滤
            return "正的小数字"
        case _:
            return "其他"
print(small_positive(2))    # ⑮ 正的小数字
print(small_positive(-1))   # ⑯ 其他

小数字
字母
其他
99
99
None
OR 变量不一致 -> SyntaxError: alternative patterns bind different names
正的小数字
其他


### 2.8 守卫条件

用 `if` 给模式加**额外条件**。三个要点：

1. 守卫在**模式匹配成功之后**求值，可以访问该模式捕获的变量
2. 守卫为假时**不执行该分支**，继续尝试下一个 case（不会直接跳出）
3. 守卫里可以写任意表达式（函数调用、字符串方法都可以）——这是模式部分做不到的

In [23]:
# 2-8 守卫条件：模式负责结构，守卫负责语义
def sign(n):
    match n:
        case x if x > 0:     # ① 先捕获到 x，再判断 x > 0
            return "正数"
        case x if x < 0:     # ② 同一个模式换个守卫
            return "负数"
        case _:              # ③ 兜底（0）
            return "零"
print(sign(5), sign(-5), sign(0))   # ④ 正数 负数 零

def region(point):
    match point:
        case (x, y) if x == y:      # ⑤ 守卫访问捕获的 x、y
            return "对角线"
        case (x, y) if x > y:
            return "下方（x 大）"
        case (x, y):
            return "上方（y 大）"
print(region((3, 3)))    # ⑥ 对角线
print(region((5, 2)))    # ⑦ 下方（x 大）
print(region((1, 4)))    # ⑧ 上方（y 大）

正数 负数 零
对角线
下方（x 大）
上方（y 大）


In [24]:
# 2-8.1 守卫失败后：继续尝试下一个 case
def route(n):
    match n:
        case x if x > 100:          # ① n=50 时模式命中但守卫失败
            return "大数"
        case x if x > 10:           # ② 继续尝试这里：模式命中、守卫成功
            return "中数"
        case _:
            return "小数"
print(route(50))    # ③ 中数 —— 证明守卫失败会继续往下走

# ④ 进阶验证：守卫失败后绑定是否保留？（CPython 实测行为）
def guard_binding_probe():
    match -5:
        case n if n > 0:      # ⑤ 模式命中（n 绑定 -5），守卫失败
            return "正数"
    # ⑥ match 语句结束（无分支执行），检查 n 是否还能访问
    try:
        return f"match 结束后 n 仍可访问, n = {n}"    # ⑦ CPython 中绑定会保留
    except NameError:
        return "n 已被清除"
print(guard_binding_probe())   # ⑧ 输出: match 结束后 n 仍可访问, n = -5
# ⑨ 结论：CPython 不会清除失败守卫留下的绑定，但这是实现细节，
#    不同 Python 实现可能不同 —— 生产代码绝不要依赖这个行为！

中数
match 结束后 n 仍可访问, n = -5


### 2.9 值模式

用**点号访问**的值做常量比较：`Color.RED`、`math.pi`。判定规则非常简单：

- 模式中出现**裸名字**（`MAX`、`pi`）→ 捕获模式
- 模式中出现**点号访问**（`Color.RED`、`math.pi`）→ 值模式（用 `==` 比较）

典型用途：**枚举分发**。这是替代“常量比较陷阱”的标准答案。

In [25]:
# 2-9 值模式：枚举与点号访问
import enum                       # ① 导入枚举模块
import math                       # ② 导入 math（后面用 math.pi）

class Color(enum.Enum):           # ③ 定义颜色枚举
    RED = 1                       # ④ 成员 RED
    GREEN = 2
    BLUE = 3

def color_name(c):
    match c:
        case Color.RED:           # ⑤ 值模式：Color.RED 是点号访问 → 比较 c == Color.RED
            return "红"
        case Color.GREEN:
            return "绿"
        case Color.BLUE:
            return "蓝"
        case _:
            return "未知"

print(color_name(Color.RED))      # ⑥ 红
print(color_name(Color.BLUE))     # ⑦ 蓝

# ⑧ 值模式不限于枚举：任何点号访问都行
def is_pi(x):
    match x:
        case math.pi:             # ⑨ math.pi 是属性访问 → 值模式（比较浮点相等）
            return "是 π"
        case _:
            return "不是 π"
print(is_pi(3.141592653589793))   # ⑩ 是 π

# ⑪ 对比：裸名字 pi 是捕获，不是比较！
def fake_pi(x):
    match x:
        case pi:                  # ⑫ 裸名字 → 捕获一切
            return f"捕获了 {pi}, 永远命中"
print(fake_pi(12345))             # ⑬ 捕获了 12345, 永远命中

红
蓝
是 π
捕获了 12345, 永远命中


### 2.10 组合模式

模式的真正威力在于**任意嵌套组合**：映射套映射、字面量套捕获、再配守卫。下面这个例子是“API 响应分发”的骨架——也是后面综合实战的预演。

In [26]:
# 2-10 组合模式：嵌套映射 + 字面量 + 捕获 + 守卫
def dispatch(data):
    match data:
        # ① 外层映射模式 + 嵌套映射 + 守卫（age 由内层捕获，守卫直接用）
        case {"type": "user", "info": {"name": name, "age": age}} if age >= 18:
            return f"成年用户: {name}"
        # ② 同样的结构，没有 age 键或未成年走这里
        case {"type": "user", "info": {"name": name}}:
            return f"未成年用户: {name}"
        # ③ 只关心 type 键的字面量
        case {"type": "guest"}:
            return "访客"
        # ④ 兜底
        case _:
            return "未知"

print(dispatch({"type": "user", "info": {"name": "Alice", "age": 30}}))  # ⑤ 成年用户: Alice
print(dispatch({"type": "user", "info": {"name": "Bob", "age": 15}}))    # ⑥ 未成年用户: Bob
print(dispatch({"type": "user", "info": {"name": "Carol"}}))             # ⑦ 未成年用户: Carol
print(dispatch({"type": "guest"}))                                       # ⑧ 访客
print(dispatch({"type": "admin", "level": 9}))                           # ⑨ 未知

成年用户: Alice
未成年用户: Bob
未成年用户: Carol
访客
未知


## 三、反例单元：亲手触发 9 个经典错误

**本章解决什么问题**：每个易错点按「错误写法（真实触发报错/异常症状）→ 正确写法 → 输出对比」三段式展开。所有报错都通过 `try/except` 捕获后打印，保证单元本身可正常执行。

| # | 反例 | 错误类型 |
|---|---|---|
| R1 | 捕获模式放在前面，吞噬后续分支 | `SyntaxError`（3.12+ 静态拦截）/ 逻辑错误（≤3.11 静默死分支） |
| R2 | `case MAX:` 误当常量比较 | 逻辑错误（静默绑定） |
| R3 | 位置类模式但没定义 `__match_args__` | `TypeError` |
| R4 | OR 模式各分支绑定不同变量 | `SyntaxError` |
| R5 | 模式中写表达式 `case 1 + 1:` | `SyntaxError` |
| R6 | 模式中写 f-string `case f"/users/{uid}":` | `SyntaxError` |
| R7 | 内置异常位置模式 `case ValueError(msg):` | `TypeError`（**源文档错误修正**） |
| R8 | 试图用序列模式解构字符串 | 逻辑错误（静默不匹配，**源文档错误修正**） |
| R9 | `case _:` 不放最后 | `SyntaxError`（3.12+ 静态拦截）/ 逻辑错误（≤3.11 死分支） |

In [27]:
# R1 错误写法：捕获模式在前，吞噬一切
# ① 经典错误版：裸捕获在前 + 具体模式在后
bad_src = "def f(x):\n match x:\n  case y:\n   return 1\n  case 0:\n   return 2"
try:
    compile(bad_src, "<demo>", "exec")            # ② 直接编译会怎样？
    print("竟然编译通过（不应该发生）")
except SyntaxError as e:                          # ③ Python 3.12+ 有静态保护
    print("错误版触发 SyntaxError:", e.msg)        # ④ name capture 'y' makes remaining patterns unreachable
    print("  —— Python 3.12 起编译器直接拦截；3.10/3.11 则是静默死分支（更危险）")

# ⑤ 更隐蔽的变体：映射捕获模式在前（只匹配含 k 键的 dict，编译器无法静态判定不可达）
def sneaky_order(d):
    match d:
        case {"k": y}:                 # ⑥ 任何含 k 键的 dict 都命中这里
            return f"泛匹配: {y}"
        case {"k": 0, "extra": 1}:     # ⑦ 更具体的模式成为死分支！
            return "精确匹配"
print("隐蔽版 sneaky_order({'k': 0, 'extra': 1}) ->", sneaky_order({"k": 0, "extra": 1}))
# ⑧ 输出: 泛匹配: 0（本想精确匹配！）—— 编译器不报错，纯靠 review 发现

# 正确写法：具体模式在前，泛化模式在后
def good_order(d):
    match d:
        case {"k": 0, "extra": 1}:     # ⑨ 具体在前
            return "精确匹配"
        case {"k": y}:                 # ⑩ 泛化兜底在后
            return f"泛匹配: {y}"
print("正确版 good_order({'k': 0, 'extra': 1}) ->", good_order({"k": 0, "extra": 1}))  # ⑪ 精确匹配
print("正确版 good_order({'k': 9})          ->", good_order({"k": 9}))                # ⑫ 泛匹配: 9

错误版触发 SyntaxError: name capture 'y' makes remaining patterns unreachable
  —— Python 3.12 起编译器直接拦截；3.10/3.11 则是静默死分支（更危险）
隐蔽版 sneaky_order({'k': 0, 'extra': 1}) -> 泛匹配: 0
正确版 good_order({'k': 0, 'extra': 1}) -> 精确匹配
正确版 good_order({'k': 9})          -> 泛匹配: 9


In [28]:
# R2 错误写法：case MAX 误当常量比较（已在 2-3.2 验证，此处聚焦对比输出）
MAX = 100    # ① 模块级常量

def bad_const(x):
    match x:
        case MAX:                        # ② 错误：裸名字 = 捕获，匹配一切
            return "命中（其实是捕获）"
print("错误版 bad_const(7) ->", bad_const(7))     # ③ x=7 明明 != 100，却命中了！

def good_const(x):
    match x:
        case 100:                        # ④ 正确一：字面量
            return "等于 100"
        case n if n == MAX:              # ⑤ 正确二：守卫 + 捕获（比较逻辑写在守卫里）
            return "等于 MAX"
        case _:
            return "都不等于"
print("正确版 good_const(7)   ->", good_const(7))    # ⑥ 都不等于
print("正确版 good_const(100) ->", good_const(100))  # ⑦ 等于 100

错误版 bad_const(7) -> 命中（其实是捕获）
正确版 good_const(7)   -> 都不等于
正确版 good_const(100) -> 等于 100


In [29]:
# R3 错误写法：位置类模式但类没定义 __match_args__
class Point:
    def __init__(self, x, y):    # ① 只有构造函数，没有 __match_args__
        self.x, self.y = x, y

try:
    match Point(1, 2):
        case Point(1, 2):        # ② 错误：位置模式需要 __match_args__ 支撑
            print("命中")
except TypeError as e:           # ③ 真实触发 TypeError
    print("错误版触发 TypeError:", e)

# 正确写法一：定义 __match_args__
class PointV2:
    __match_args__ = ("x", "y")  # ④ 声明位置顺序
    def __init__(self, x, y):
        self.x, self.y = x, y

match PointV2(1, 2):
    case PointV2(1, 2):          # ⑤ 位置模式现在可用
        print("正确版（__match_args__）: 命中")

# 正确写法二：改用关键字模式（无需 __match_args__）
match Point(1, 2):
    case Point(x=1, y=2):        # ⑥ 关键字模式按属性名匹配
        print("正确版（关键字模式）: 命中")

错误版触发 TypeError: Point() accepts 0 positional sub-patterns (2 given)
正确版（__match_args__）: 命中
正确版（关键字模式）: 命中


In [30]:
# R4 错误写法：OR 模式各分支绑定不同的变量
bad_src = "match x:\n    case [1, a] | [b, 2]:\n        pass"   # ① a 和 b 不一致
try:
    compile(bad_src, "<demo>", "exec")    # ② 编译期直接报错
    print("竟然编译通过（不应该发生）")
except SyntaxError as e:                  # ③ 捕获语法错误
    print("错误版触发 SyntaxError:", e.msg)

# 正确写法：所有分支绑定相同变量
def good_or(pair):
    match pair:
        case [1, v] | [v, 1]:   # ④ 两个分支都只绑定 v
            return v
        case _:
            return None
print("正确版 good_or([1, 99]) ->", good_or([1, 99]))   # ⑤ 99
print("正确版 good_or([99, 1]) ->", good_or([99, 1]))   # ⑥ 99

错误版触发 SyntaxError: alternative patterns bind different names
正确版 good_or([1, 99]) -> 99
正确版 good_or([99, 1]) -> 99


In [31]:
# R5 错误写法：模式中出现表达式
try:
    compile("match x:\n    case 1 + 1:\n        pass", "<demo>", "exec")   # ① 表达式不是模式
    print("竟然编译通过（不应该发生）")
except SyntaxError as e:
    print("错误版触发 SyntaxError:", e.msg)     # ② patterns may only match literals and attribute lookups

# 正确写法：字面量 / 守卫里做运算
def match_two(x):
    match x:
        case 2:                        # ③ 直接写字面量结果
            return "是 2"
        case n if n == 1 + 1:          # ④ 或者捕获后在守卫里算
            return "等于 1+1"
        case _:
            return "其他"
print("正确版 match_two(2) ->", match_two(2))   # ⑤ 是 2

错误版触发 SyntaxError: imaginary number required in complex literal
正确版 match_two(2) -> 是 2


In [32]:
# R6 错误写法：模式中写 f-string 做“路径参数提取”
bad_src = 'match req:\n    case {"path": f"/users/{uid}"}:\n        pass'   # ① 模式不支持插值
try:
    compile(bad_src, "<demo>", "exec")
    print("竟然编译通过（不应该发生）")
except SyntaxError as e:
    print("错误版触发 SyntaxError:", e.msg)     # ② SyntaxError: patterns may only match literals and attribute lookups

# 正确写法：守卫 + 字符串方法
def get_user_route(req):
    match req:
        case {"method": "GET", "path": path} if path.startswith("/users/"):   # ③ 守卫做前缀判断
            uid = path.removeprefix("/users/")                                # ④ 在分支体内提取参数
            return f"查询用户 {uid}"
        case _:
            return "其他路由"
print(get_user_route({"method": "GET", "path": "/users/42"}))   # ⑤ 查询用户 42

错误版触发 SyntaxError: patterns may only match literals and attribute lookups
查询用户 42


In [33]:
# R7 错误写法：内置异常用位置模式提取消息（【源文档修正说明】）
# 源文档 11.9 写了 case ValueError(msg): —— 实际会 TypeError！
# 原因：内置异常类没有定义 __match_args__，位置子模式不可用。

def handle(e):
    try:
        match e:
            case ValueError(msg):          # ① 错误写法：位置模式
                return f"值错误: {msg}"
            case _:
                return "其他"
    except TypeError as te:                # ② 真实触发 TypeError
        return f"错误版触发 TypeError: {te}"

print(handle(ValueError("boom")))          # ③ 输出 TypeError 信息

# 正确写法一：空类模式 + as 捕获整个异常，再取 args
def handle_v1(e):
    match e:
        case ValueError() as err:          # ④ 只做类型检查，as 绑定异常对象
            return f"值错误: {err.args[0] if err.args else ''}"
        case _:
            return "其他"

# 正确写法二：内置异常支持关键字属性模式（OSError 的 errno 是属性）
def handle_v2(e):
    match e:
        case OSError(errno=2):             # ⑤ 关键字模式按属性匹配，可用！
            return "文件不存在"
        case OSError(errno=errno_val) if errno_val == 13:   # ⑥ 属性绑定 + 守卫
            return "权限不足"
        case OSError() as err:
            return f"其他 OS 错误: {err}"
        case _:
            return "其他"

print(handle_v1(ValueError("boom")))                     # ⑦ 值错误: boom
print(handle_v2(OSError(2, "no such file")))             # ⑧ 文件不存在
print(handle_v2(OSError(13, "permission denied")))       # ⑨ 权限不足

错误版触发 TypeError: ValueError() accepts 0 positional sub-patterns (1 given)
值错误: boom
文件不存在
权限不足


In [34]:
# R8 错误写法：试图用序列模式解构字符串（【源文档修正说明】）
# 源文档 4.3 称“字符串也可以按字符解构”——实际 str/bytes/bytearray 被序列模式排除。

def bad_str_match(s):
    match s:
        case [first, second]:              # ① 想解构前两个字符
            return f"解构成功: {first}{second}"
        case _:
            return "未匹配（静默失败！）"    # ② 字符串永远走这里 —— 没有任何报错

print("错误版 bad_str_match('hi') ->", bad_str_match("hi"))   # ③ 未匹配 —— 静默不匹配，极易埋雷

# 正确写法：字面量 / 守卫 / 先显式转 list
def good_str_match(s):
    match s:
        case "hi":                         # ④ 字面量整体比较
            return "问候语"
        case t if isinstance(t, str) and len(t) == 2:   # ⑤ 守卫做长度/内容判断
            return f"两个字符: {t}"
        case _:
            return "其他"

print("正确版 good_str_match('hi') ->", good_str_match("hi"))    # ⑥ 问候语
print("正确版 good_str_match('ok') ->", good_str_match("ok"))    # ⑦ 两个字符: ok

# ⑧ 如果确实要按字符“解构”：先显式转成列表
match list("hi"):
    case [c1, c2]:                         # ⑨ subject 已经是 list，可以正常序列匹配
        print("转 list 后解构:", c1, c2)    # ⑩ h i

错误版 bad_str_match('hi') -> 未匹配（静默失败！）
正确版 good_str_match('hi') -> 问候语
正确版 good_str_match('ok') -> 两个字符: ok
转 list 后解构: h i


In [35]:
# R9 错误写法：case _ 不放最后，制造死分支
# ① 经典错误版：通配符在最前
bad_src = "def f(x):\n match x:\n  case _:\n   return 1\n  case 1:\n   return 2"
try:
    compile(bad_src, "<demo>", "exec")
    print("竟然编译通过（不应该发生）")
except SyntaxError as e:                          # ② Python 3.12+ 静态拦截
    print("错误版触发 SyntaxError:", e.msg)        # ③ wildcard makes remaining patterns unreachable

# ④ 隐蔽变体：定长序列通配 [_, _, _] 在前（编译器无法静态判定，3 元素列表全被吞）
def sneaky_wildcard(seq):
    match seq:
        case [_, _, _]:               # ⑤ 匹配任意 3 元素序列
            return "三元素分支"
        case [1, 2, 3]:               # ⑥ 死分支：[1,2,3] 早就被上面截胡
            return "精确的 1,2,3"
print("隐蔽版 sneaky_wildcard([1, 2, 3]) ->", sneaky_wildcard([1, 2, 3]))  # ⑦ 三元素分支

def good_wildcard(seq):
    match seq:
        case [1, 2, 3]:               # ⑧ 精确在前
            return "精确的 1,2,3"
        case [_, _, _]:               # ⑨ 泛化在后
            return "三元素分支"
        case _:
            return "其他"
print("正确版 good_wildcard([1, 2, 3]) ->", good_wildcard([1, 2, 3]))   # ⑩ 精确的 1,2,3
print("正确版 good_wildcard([7, 8, 9]) ->", good_wildcard([7, 8, 9]))   # ⑪ 三元素分支
print("正确版 good_wildcard([1])       ->", good_wildcard([1]))         # ⑫ 其他

错误版触发 SyntaxError: wildcard makes remaining patterns unreachable
隐蔽版 sneaky_wildcard([1, 2, 3]) -> 三元素分支
正确版 good_wildcard([1, 2, 3]) -> 精确的 1,2,3
正确版 good_wildcard([7, 8, 9]) -> 三元素分支
正确版 good_wildcard([1])       -> 其他


## 四、实用示例集：5 个工程高频模式

**本章解决什么问题**：正文讲的是“语法怎么用”，本章讲“工程上什么时候用”。每个示例标注适用场景与注意点，并尽量取材于 **LLM API 网关** 业务场景（本文默认业务场景：*构建 LLM API 网关，需要解析大模型返回的 JSON 响应、分发工具调用、管理任务状态*）。

| # | 模式 | 适用场景 | 关键注意点 |
|---|---|---|---|
| P1 | API 响应分发 | 处理外部/LLM 服务返回的结构化 JSON | 映射模式是部分匹配，多出的键不影响 |
| P2 | 命令解析 | 斜杠命令 / REPL / 聊天指令 | `split()` 后接序列模式，长度即参数个数 |
| P3 | 递归求值（AST） | 表达式计算、配置继承、消息树遍历 | 递归 + match 是天作之合，注意终止条件 |
| P4 | dataclass 多态分发 | 领域对象处理（图形、事件、消息） | dataclass 自动生成 `__match_args__` |
| P5 | 状态机 | 任务/会话生命周期管理 | 元组 `(state, event)` + OR 模式合并等价迁移 |

In [36]:
# P1 API 响应分发：LLM 网关处理模型返回（映射模式 + 守卫 + 嵌套）
class NotFound(Exception):        # ① 业务异常：资源不存在
    pass

class ServerError(Exception):     # ② 业务异常：上游服务错误
    pass

class ClientError(Exception):     # ③ 业务异常：客户端错误
    pass

def handle_response(resp):        # ④ resp 是上游 LLM 服务的 JSON 响应（dict）
    match resp:
        case {"status": 200, "data": data}:        # ⑤ 成功：提取 data 载荷
            return data
        case {"status": 404}:                      # ⑥ 404：无 data 也能精确命中
            raise NotFound()
        case {"status": 500, "error": msg}:        # ⑦ 服务端错误：嵌套提取 error 消息
            raise ServerError(msg)
        case {"status": code} if code >= 400:      # ⑧ 其他 4xx：守卫兜数值范围
            raise ClientError(code)
        case _:                                    # ⑨ 未知结构
            raise ValueError("未知响应")

# ⑩ 逐一验证五种分支
print("成功:", handle_response({"status": 200, "data": {"tokens": 42}}))   # ⑪ {'tokens': 42}
for resp, exc in [
    ({"status": 404}, NotFound),
    ({"status": 500, "error": "model overloaded"}, ServerError),
    ({"status": 429}, ClientError),
]:
    try:
        handle_response(resp)
    except exc as e:                             # ⑫ 按预期异常类型捕获
        print(f"预期异常 {exc.__name__}: {e}")

成功: {'tokens': 42}
预期异常 NotFound: 
预期异常 ServerError: model overloaded
预期异常 ClientError: 429


In [37]:
# P2 命令解析：LLM 网关的运维斜杠命令（序列模式，长度即参数个数）
def parse_command(cmd):                        # ① 输入是一行命令字符串
    match cmd.split():                         # ② 先 split 成 token 列表，再做序列匹配
        case ["go", direction]:                # ③ 恰好 2 个 token 且第一个是 go
            return f"向 {direction} 走"
        case ["look"]:                         # ④ 恰好 1 个 token
            return "观察四周"
        case ["take", item]:                   # ⑤ 2 个 token，第 1 个是 take
            return f"捡起 {item}"
        case ["use", item, "on", target]:      # ⑥ 4 个 token，第 1/3 个必须是字面量
            return f"用 {item} 对 {target}"
        case ["deploy", model, *args]:         # ⑦ 变长：/deploy chatglm --replicas 3
            return f"部署模型 {model}, 附加参数 {args}"
        case _:                                # ⑧ 兜底
            return "无效命令"

print(parse_command("go north"))            # ⑨ 向 north 走
print(parse_command("use key on door"))     # ⑩ 用 key 对 door
print(parse_command("deploy chatglm --replicas 3"))   # ⑪ 部署模型 chatglm, 附加参数 ['--replicas', '3']
print(parse_command("hello??"))             # ⑫ 无效命令

向 north 走
用 key 对 door
部署模型 chatglm, 附加参数 ['--replicas', '3']
无效命令


In [38]:
# P3 递归求值：表达式 AST（嵌套序列模式 + 递归）
def evaluate(node):                            # ① node 是 ('op', 子节点...) 形式的元组树
    match node:
        case ("num", value):                   # ② 叶子节点：数字字面量
            return value
        case ("add", left, right):             # ③ 加法：递归求左右子树
            return evaluate(left) + evaluate(right)
        case ("mul", left, right):             # ④ 乘法
            return evaluate(left) * evaluate(right)
        case ("neg", operand):                 # ⑤ 取负
            return -evaluate(operand)
        case _:                                # ⑥ 未知节点类型
            raise ValueError(f"未知节点: {node}")

# ⑦ 构造表达式: 1 + (2 * 3) = 7
expr = ("add", ("num", 1), ("mul", ("num", 2), ("num", 3)))
print("1 + 2*3 =", evaluate(expr))             # ⑧ 7

# ⑨ 更复杂的: -(1 + 2) * 3 = -9
expr2 = ("mul", ("neg", ("add", ("num", 1), ("num", 2))), ("num", 3))
print("-(1+2)*3 =", evaluate(expr2))           # ⑩ -9

1 + 2*3 = 7
-(1+2)*3 = -9


In [39]:
# P4 dataclass 多态分发：几何图形面积（类模式 + 嵌套 + __match_args__）
from dataclasses import dataclass      # ① dataclass 会自动生成 __match_args__！

@dataclass
class Point:                           # ② 字段顺序即位置模式顺序: (x, y)
    x: float
    y: float

@dataclass
class Circle:                          # ③ 位置顺序: (center, radius)
    center: Point
    radius: float

@dataclass
class Rectangle:                       # ④ 位置顺序: (top_left, bottom_right)
    top_left: Point
    bottom_right: Point

def area(shape):                       # ⑤ 一个函数处理三种图形
    match shape:
        case Circle(radius=r):                  # ⑥ 位置模式 + 捕获半径
            return 3.14159 * r * r
        case Rectangle(Point(x=x1, y=y1), Point(x=x2, y=y2)):   # ⑦ 双重嵌套：图形套点
            return abs((x2 - x1) * (y2 - y1))
        case _:
            raise ValueError("未知图形")

print("圆面积:", round(area(Circle(Point(0, 0), 5)), 2))            # ⑧ 78.54
print("矩形面积:", area(Rectangle(Point(0, 0), Point(3, 4))))      # ⑨ 12
print("验证 dataclass 自动 __match_args__:", Point.__match_args__) # ⑩ ('x', 'y')

圆面积: 78.54
矩形面积: 12
验证 dataclass 自动 __match_args__: ('x', 'y')


In [40]:
# P5 状态机：LLM 任务生命周期（元组模式 + OR 模式合并等价迁移）
def next_state(state, event):              # ① 输入当前状态 + 事件，输出下一状态
    match (state, event):                  # ② 把两个值打包成元组一起匹配
        case ("idle", "start"):            # ③ 空闲 + 启动 → 运行
            return "running"
        case ("running", "pause"):         # ④ 运行 + 暂停 → 暂停
            return "paused"
        case ("paused", "resume"):         # ⑤ 暂停 + 恢复 → 运行
            return "running"
        case ("running", "stop") | ("paused", "stop"):   # ⑥ OR 模式：两种状态都接受 stop
            return "idle"
        case _:                             # ⑦ 非法迁移：保持原状态
            return state

# ⑧ 完整生命周期走一遍
s = "idle"
for event in ["start", "pause", "resume", "stop"]:
    s = next_state(s, event)
    print(f"{event:8} -> {s}")             # ⑨ 输出每步迁移结果
# ⑩ 非法迁移验证
print("running 时 pause 之后又 pause:", next_state("paused", "pause"))   # ⑪ paused（保持不变）

start    -> running
pause    -> paused
resume   -> running
stop     -> idle
running 时 pause 之后又 pause: paused


## 五、内置 / 生态能力对照

**本章解决什么问题**：match 不是孤岛——它和标准库的枚举、dataclass、命名元组深度协同，也要和 if-elif、字典分发等“竞品”划清边界。

### 5.1 标准库协同

| 标准库能力 | 与 match 的关系 | 关键点 |
|---|---|---|
| `enum.Enum` | 提供**值模式**的最佳搭档 | `case Color.RED:` 是比较；裸写 `RED:` 是捕获 |
| `dataclasses.dataclass` | 自动生成 `__match_args__` | 普通类要手写；dataclass 免写 |
| `typing.NamedTuple` | 自动生成 `__match_args__` | 且本身是元组，还支持序列模式 |
| `ast` 模块 | `ast.match_case` / `ast.Match` 节点 | 程序化分析含 match 的代码时用 |
| `math` / 模块属性 | 点号访问即值模式 | `case math.pi:` 是比较 |
| 内置异常 | **没有** `__match_args__` | 位置模式不可用，只能 `cls()` 或关键字属性 |

### 5.2 分发机制竞品对比

| 维度 | `match` | `if-elif` | 字典分发 `{'a': fn}[key]` | 访问者模式 |
|---|---|---|---|---|
| 结构解构 | ✅ 原生支持 | ❌ 手动 | ❌ 不支持 | ✅（需类体系） |
| 类型检查 | ✅ 类模式 | `isinstance()` | ❌ | ✅ |
| 变量绑定 | ✅ 自动 | ❌ 手动 | ❌ | ✅ |
| 守卫条件 | ✅ `if` | ✅ 天然 | ❌ 需包一层 | ✅ |
| 键须先注册 | 否 | 否 | **是**（KeyError 风险） | 否 |
| 简单值比较 | 可用但重 | ✅ 最合适 | ✅ 高频调用更快 | 杀鸡用牛刀 |
| Python 版本 | 3.10+ | 全版本 | 全版本 | 全版本 |

In [41]:
# 5-1 验证：NamedTuple 同时支持序列模式与位置类模式
from typing import NamedTuple

class Vec(NamedTuple):        # ① NamedTuple 自动生成 __match_args__
    x: float
    y: float

match Vec(3, 4):              # ② subject 是 NamedTuple 实例
    case Vec(a, b):           # ③ 类位置模式可用（自动 __match_args__）
        print("类位置模式:", a, b)    # ④ 3 4

match Vec(3, 4):
    case [a, b]:              # ⑤ 它也是元组 → 序列模式同样可用！
        print("序列模式:", a, b)      # ⑥ 3 4

# ⑦ 字典分发 vs match：字典要求键预先注册
handlers = {"ping": lambda: "pong", "status": lambda: "ok"}
print("字典分发:", handlers.get("ping", lambda: "unknown")())   # ⑧ pong

# ⑨ match 不需要预注册，天然支持“结构+值”的复合条件
def route(msg):
    match msg:
        case {"cmd": "ping", "nonce": n} if n > 0:   # ⑩ 字典分发很难表达这种复合条件
            return "pong(有效nonce)"
        case {"cmd": "ping"}:
            return "pong"
        case _:
            return "unknown"
print("match 分发:", route({"cmd": "ping", "nonce": 7}))   # ⑪ pong(有效nonce)

类位置模式: 3 4
序列模式: 3 4
字典分发: pong
match 分发: pong(有效nonce)


## 六、综合实战：LLM API 网关的响应处理管道

**本章解决什么问题**：把前面 10 类模式中的 6 类（映射、类、值、OR、守卫、组合）拧成一根真实可用的管道。

> **业务场景说明**：由于需求模板中的 `{{业务场景描述}}` 未填写，本文根据项目路径（`LLMOps/llmops-api`）**假设业务场景为**：*LLM API 网关服务——接收上游大模型的 JSON 响应，归一化为内部事件对象，按事件类型分发处理，并驱动会话任务状态机*。若实际场景不同，管道结构可直接套用。

### 设计蓝图（数据流）

```
上游 JSON 响应
    │
    ▼ ① 映射模式 + 守卫：归一化（dict → 内部事件 dataclass）
normalize() ── TextChunk / ToolCall / UsageReport / ErrorResponse
    │
    ▼ ② 类模式 + 值模式 + OR 模式：分发处理
dispatch()  ── 按事件类型路由到不同 handler
    │
    ▼ ③ 元组模式 + OR 模式：状态机
advance()   ── 会话任务生命周期迁移
```

### 设计取舍（为什么这样分层）

| 决策 | 备选方案 | 取舍理由 |
|---|---|---|
| 第一层用**映射模式**归一化，不用类模式 | 直接 match 原始 dict 到处传 | 上游 JSON 是不可控的异构数据，先收敛成内部类型，后续代码只面对受控类型 |
| 第二层用**类模式**分发，不用字典分发 | `{TextChunk: fn}` 查表 | 类模式可以带结构约束（如 `ToolCall(name="search")`）并自动绑定字段，字典查表做不到 |
| 错误码用**枚举值模式**，不用裸整数 | `case 429:` | 裸整数可读性差且容易和别处魔法数字撞车；`ErrCode.RATE_LIMIT` 自文档化 |
| 等价迁移用 **OR 模式**合并 | 写多条重复 case | `("running","stop") | ("paused","stop")` 一行表达“任意活动状态可停止” |
| 状态机匹配**元组** `(state, event)` | 嵌套 match | 元组模式让「状态×事件」矩阵一目了然，新增迁移只加一行 |

In [42]:
# 6-1 第一层：归一化 —— 映射模式 + 嵌套 + 守卫，把异构 JSON 收敛为内部事件
from dataclasses import dataclass
import enum

# ① 内部事件类型（dataclass 自动生成 __match_args__，供第二层类模式使用）
@dataclass
class TextChunk:
    content: str                      # ② 增量文本片段

@dataclass
class ToolCall:
    name: str                         # ③ 工具名，如 "search"
    arguments: dict                   # ④ 工具参数

@dataclass
class UsageReport:
    prompt_tokens: int                # ⑤ 输入 token 数
    completion_tokens: int            # ⑥ 输出 token 数

class ErrCode(enum.Enum):             # ⑦ 错误码枚举：第二层用值模式匹配
    RATE_LIMIT = "rate_limit"
    MODEL_OVERLOADED = "model_overloaded"
    INVALID_REQUEST = "invalid_request"
    UNKNOWN = "unknown"

@dataclass
class ErrorResponse:
    code: ErrCode                     # ⑧ 结构化错误码
    message: str

def normalize(resp):                  # ⑨ 上游 JSON → 内部事件
    match resp:
        # ⑩ 文本增量：OpenAI 风格 choices[0].delta.content
        case {"object": "chat.completion.chunk",
              "choices": [{"delta": {"content": str(content)}}]}:
            return TextChunk(content)
        # ⑪ 工具调用：tool_calls[0].function 的 name + arguments（JSON 串）
        case {"choices": [{"delta": {"tool_calls": [
                {"function": {"name": name, "arguments": str(args)}}]}}]}:
            return ToolCall(name, {"raw": args})
        # ⑫ 用量上报：嵌套取 usage 两个字段
        case {"usage": {"prompt_tokens": int(p), "completion_tokens": int(c)}}:
            return UsageReport(p, c)
        # ⑬ 错误响应：code 映射为枚举（守卫 + OR 值模式）
        case {"error": {"code": "429" | "rate_limit", "message": msg}}:
            return ErrorResponse(ErrCode.RATE_LIMIT, msg)
        case {"error": {"code": "model_overloaded", "message": msg}}:
            return ErrorResponse(ErrCode.MODEL_OVERLOADED, msg)
        case {"error": {"code": _, "message": msg}}:
            return ErrorResponse(ErrCode.UNKNOWN, msg)
        case _:
            return ErrorResponse(ErrCode.UNKNOWN, "无法识别的响应结构")

# ⑭ 四类真实响应逐一验证
chunks = normalize({"object": "chat.completion.chunk",
                    "choices": [{"delta": {"content": "你好"}}]})
tools = normalize({"choices": [{"delta": {"tool_calls": [
    {"function": {"name": "search", "arguments": '{"q": "match语句"}'}}]}}]})
usage = normalize({"usage": {"prompt_tokens": 120, "completion_tokens": 35}})
error = normalize({"error": {"code": "rate_limit", "message": "QPS 超限"}})
print("文本事件:  ", chunks)      # ⑮ TextChunk(content='你好')
print("工具事件:  ", tools)       # ⑯ ToolCall(name='search', ...)
print("用量事件:  ", usage)       # ⑰ UsageReport(prompt_tokens=120, completion_tokens=35)
print("错误事件:  ", error)       # ⑱ ErrorResponse(code=<ErrCode.RATE_LIMIT...>, ...)

文本事件:   TextChunk(content='你好')
工具事件:   ToolCall(name='search', arguments={'raw': '{"q": "match语句"}'})
用量事件:   UsageReport(prompt_tokens=120, completion_tokens=35)
错误事件:   ErrorResponse(code=<ErrCode.RATE_LIMIT: 'rate_limit'>, message='QPS 超限')


In [43]:
# 6-2 第二层：分发处理 —— 类模式 + 值模式 + 守卫，按事件类型路由
def dispatch(event):                          # ① 输入是第一层产出的内部事件
    match event:
        case TextChunk(content):              # ② 位置类模式：绑定 content
            return f"[流式输出] 追加文本: {content!r}"

        case ToolCall(name="search", arguments=args):     # ③ 关键字模式：只关心搜索工具
            return f"[工具调用] 执行检索, 参数: {args}"

        case ToolCall(name=name):             # ④ 其他工具走通用执行器
            return f"[工具调用] 通用执行 {name}"

        case UsageReport(prompt_tokens=p, completion_tokens=c) if p + c > 100_000:
            return f"[用量告警] 总 token {p+c:,} 超阈值，触发熔断"    # ⑤ 守卫做阈值判断

        case UsageReport(prompt_tokens=p, completion_tokens=c):
            return f"[用量记录] 输入 {p} + 输出 {c} tokens"

        # ⑥ 值模式匹配枚举错误码 + 守卫细分
        case ErrorResponse(code=ErrCode.RATE_LIMIT, message=msg):
            return f"[错误] 限流，进入退避重试: {msg}"
        case ErrorResponse(code=ErrCode.MODEL_OVERLOADED):
            return "[错误] 模型过载，切换备用节点"
        case ErrorResponse(code=ErrCode.INVALID_REQUEST, message=msg):
            return f"[错误] 请求不合法，直接拒绝: {msg}"
        case ErrorResponse():                 # ⑦ 未知错误兜底
            return "[错误] 未知错误，人工介入"

# ⑧ 逐一验证分发结果
for ev in [chunks, tools, usage,
           UsageReport(90_000, 20_000),       # ⑨ 超阈值用量 → 告警分支
           error,
           ErrorResponse(ErrCode.MODEL_OVERLOADED, "busy")]:
    print(dispatch(ev))

[流式输出] 追加文本: '你好'
[工具调用] 执行检索, 参数: {'raw': '{"q": "match语句"}'}
[用量记录] 输入 120 + 输出 35 tokens
[用量告警] 总 token 110,000 超阈值，触发熔断
[错误] 限流，进入退避重试: QPS 超限
[错误] 模型过载，切换备用节点


In [44]:
# 6-3 第三层：状态机 —— 元组模式 + OR 模式，驱动会话任务生命周期
def advance(state, event):                       # ① (状态, 事件) → 新状态
    match (state, event):
        case ("idle", "request"):                # ② 空闲 + 收到请求 → 流式中
            return "streaming"
        case ("streaming", "tool_call"):         # ③ 流式中触发工具 → 等待工具结果
            return "awaiting_tool"
        case ("awaiting_tool", "tool_result"):   # ④ 工具返回 → 继续流式
            return "streaming"
        case ("streaming", "done") | ("awaiting_tool", "done"):   # ⑤ OR：两状态均可完成
            return "idle"
        case ("streaming", "abort") | ("awaiting_tool", "abort"): # ⑥ OR：均可中止
            return "cancelled"
        case _:                                   # ⑦ 非法迁移：显式报错而不是静默吞掉
            raise ValueError(f"非法迁移: {state} + {event}")

# ⑧ 完整会话轨迹：请求 → 流式 → 工具调用 → 工具结果 → 完成
trace = [("idle", "request"), ("streaming", "tool_call"),
         ("awaiting_tool", "tool_result"), ("streaming", "done")]
state = "idle"
for ev_pair in trace:
    state = advance(state, ev_pair[1])
    print(f"{ev_pair[0]:>14} --{ev_pair[1]:<12}--> {state}")

# ⑨ 非法迁移验证（比如已完成后再 request）
try:
    advance("idle", "tool_result")               # ⑩ idle 不接受 tool_result
except ValueError as e:
    print("非法迁移被拦截:", e)

          idle --request     --> streaming
     streaming --tool_call   --> awaiting_tool
 awaiting_tool --tool_result --> streaming
     streaming --done        --> idle
非法迁移被拦截: 非法迁移: idle + tool_result


### 综合实战小结

这条 30 行的管道用到了：**映射模式**（归一化 4 种 JSON 形态）、**嵌套模式**（三层深的 `choices[0].delta`）、**类模式**（事件分发）、**关键字/位置子模式**（字段绑定）、**值模式**（枚举错误码）、**OR 模式**（错误码合并 + 状态机等价迁移）、**守卫**（token 阈值告警）。

核心设计思想一句话：**外层 match 收敛异构输入，内层 match 分发受控类型，守卫承载业务阈值，枚举消灭魔法字符串**。

## 七、常见陷阱速查表（陷阱 / 症状 / 解法）

**本章解决什么问题**：把散落全文的坑集中成一张表，每条都附最小复现代码（在下方单元中统一执行，输出即症状）。

| # | 陷阱 | 症状 | 解法 |
|---|---|---|---|
| 1 | 捕获模式放在具体模式之前 | 3.12+：`SyntaxError: name capture ... unreachable`；≤3.11：静默死分支，返回值出乎意料 | 具体模式在前，捕获/`_` 兜底在最后 |
| 2 | `case MAX:` 当常量比较 | 任何值都命中，常量被静默覆盖绑定 | `case 100:` 或 `case n if n == MAX:` |
| 3 | 位置类模式无 `__match_args__` | `TypeError: ... accepts 0 positional sub-patterns` | 定义 `__match_args__`、用 dataclass、或改关键字模式 |
| 4 | OR 模式分支绑定不同变量 | `SyntaxError: alternative patterns bind different names` | 各分支绑定同一组变量 |
| 5 | 模式中写表达式 `1 + 1` | `SyntaxError: patterns may only match literals...` | 字面量写结果，或运算放守卫 |
| 6 | 模式中写 f-string 提参数 | 同上 SyntaxError | 守卫 `if path.startswith(...)` + 分支体内提取 |
| 7 | 内置异常位置模式 `ValueError(msg)` | `TypeError: ValueError() accepts 0 positional sub-patterns` | `case ValueError() as e:` 取 `e.args`；OSError 可用关键字属性 |
| 8 | 用序列模式匹配 str/bytes | 静默不匹配，无任何报错 | 字面量/守卫；确需解构先 `list(s)` |
| 9 | `case _` 不放最后 | 其后分支永不可达 | 通配符永远最后 |
| 10 | 以为映射模式要求键全匹配 | 多余键被忽略，命中了“不该命中”的数据 | 需要严格匹配时加守卫 `if set(d) == {...}` |
| 11 | 忽略序列/映射匹配规则差异 | 序列要长度一致，映射只要键包含——搞混导致意外命中/不命中 | 记忆：**序列看长度，映射看包含** |
| 12 | 父类模式放在子类模式之前 | 子类实例永远命中父类分支 | 子类模式在前，父类在后（ specificity 降序） |

In [45]:
# 7-1 陷阱统一复现（每条都真实执行，输出即“症状”）
def trap(n, desc, fn):
    print(f"陷阱{n:02d} [{desc}]")

# 陷阱 1：捕获在前 → 死分支（3.12+ 编译器拦截；更隐蔽的映射变体在 R1 单元演示）
trap(1, "捕获吞噬后续分支", None)
try:
    compile("def t1(x):\n match x:\n  case y:\n   return 1\n  case 0:\n   return 2", "<d>", "exec")
except SyntaxError as e:
    print("  症状: SyntaxError:", e.msg, "（3.10/3.11 则静默死分支）")

# 陷阱 2：case MAX 是捕获
MAX = 100
def t2(x):
    match x:
        case MAX: return "命中"
trap(2, "常量被当捕获", None)
print("  症状: t2(7) =", t2(7), "（7 != 100 却命中）")

# 陷阱 3：无 __match_args__ 的位置模式
class P3:
    def __init__(self): self.x = 1
trap(3, "位置类模式缺 __match_args__", None)
try:
    match P3():
        case P3(1): pass
except TypeError as e:
    print("  症状: TypeError:", e)

# 陷阱 4：OR 变量不一致
trap(4, "OR 模式变量不一致", None)
try:
    compile("match x:\n case [1,a]|[b,2]:\n  pass", "<d>", "exec")
except SyntaxError as e:
    print("  症状: SyntaxError:", e.msg)

# 陷阱 5：模式中写表达式
trap(5, "模式中出现表达式", None)
try:
    compile("match x:\n case 1+1:\n  pass", "<d>", "exec")
except SyntaxError as e:
    print("  症状: SyntaxError:", e.msg)

# 陷阱 6：f-string 模式
trap(6, "f-string 模式", None)
try:
    compile('match r:\n case {"p": f"/u/{id}"}:\n  pass', "<d>", "exec")
except SyntaxError as e:
    print("  症状: SyntaxError:", e.msg)

# 陷阱 7：内置异常位置模式
trap(7, "内置异常位置模式", None)
try:
    match ValueError("x"):
        case ValueError(m): pass
except TypeError as e:
    print("  症状: TypeError:", e)

# 陷阱 8：序列模式 vs str
trap(8, "str 不被序列模式匹配", None)
r8 = "未匹配"
match "ab":
    case [a, b]: r8 = f"匹配 {a}{b}"
print("  症状:", r8, "（无任何报错，静默失败）")

# 陷阱 9：case _ 不在最后（3.12+ 编译器拦截；定长序列通配的隐蔽变体在 R9 单元演示）
trap(9, "通配符不在最后", None)
try:
    compile("def t9(x):\n match x:\n  case _:\n   return 1\n  case 1:\n   return 2", "<d>", "exec")
except SyntaxError as e:
    print("  症状: SyntaxError:", e.msg)

# 陷阱 10：映射模式部分匹配
trap(10, "映射只查指定键", None)
r10 = "未命中"
match {"a": 1, "b": 999}:
    case {"a": 1}: r10 = "命中（b 被忽略）"
print("  症状:", r10)

# 陷阱 11：序列要长度一致 vs 映射只要包含
trap(11, "序列/映射规则差异", None)
r11a = r11b = "未命中"
match (1, 2):
    case [1, 2, 3]: r11a = "命中"
match {"k": 1, "extra": 2}:
    case {"k": 1}: r11b = "命中"
print(f"  症状: 序列多一个元素 -> {r11a}；映射多一个键 -> {r11b}")

# 陷阱 12：父类模式在子类之前
class Animal: pass
class Dog(Animal): pass
def t12(o):
    match o:
        case Animal(): return "动物分支"
        case Dog(): return "狗分支"
trap(12, "父类模式在前", None)
print("  症状: t12(Dog()) =", t12(Dog()), "（期望 狗分支）")

陷阱01 [捕获吞噬后续分支]
  症状: SyntaxError: name capture 'y' makes remaining patterns unreachable （3.10/3.11 则静默死分支）
陷阱02 [常量被当捕获]
  症状: t2(7) = 命中 （7 != 100 却命中）
陷阱03 [位置类模式缺 __match_args__]
  症状: TypeError: P3() accepts 0 positional sub-patterns (1 given)
陷阱04 [OR 模式变量不一致]
  症状: SyntaxError: alternative patterns bind different names
陷阱05 [模式中出现表达式]
  症状: SyntaxError: imaginary number required in complex literal
陷阱06 [f-string 模式]
  症状: SyntaxError: patterns may only match literals and attribute lookups
陷阱07 [内置异常位置模式]
  症状: TypeError: ValueError() accepts 0 positional sub-patterns (1 given)
陷阱08 [str 不被序列模式匹配]
  症状: 未匹配 （无任何报错，静默失败）
陷阱09 [通配符不在最后]
  症状: SyntaxError: wildcard makes remaining patterns unreachable
陷阱10 [映射只查指定键]
  症状: 命中（b 被忽略）
陷阱11 [序列/映射规则差异]
  症状: 序列多一个元素 -> 未命中；映射多一个键 -> 命中
陷阱12 [父类模式在前]
  症状: t12(Dog()) = 动物分支 （期望 狗分支）


## 八、总结

**本章解决什么问题**：把全篇压缩成可复制的模板、可背诵的口诀和可自测的练习。

### 8.1 通用代码模板（直接复制改造）

```python
match subject:
    # ① 字面量：精确值分发
    case 200:
        ...
    # ② 值模式：枚举/常量（必须点号访问）
    case Color.RED:
        ...
    # ③ 序列模式：定长解构 + 变长收集
    case [op, first, *rest]:
        ...
    # ④ 映射模式：部分键匹配 + 嵌套 + 双星收集
    case {"type": "user", "profile": {"name": name}, **extra}:
        ...
    # ⑤ 类模式：类型 + 结构（dataclass 自动 __match_args__）
    case ToolCall(name="search", arguments=args):
        ...
    # ⑥ OR 模式（各分支绑定必须一致）+ 守卫（业务条件放这里）
    case ErrCode.RATE_LIMIT | ErrCode.OVERLOADED if retryable:
        ...
    # ⑦ 兜底：通配符永远最后
    case _:
        ...
```

### 8.2 记忆口诀

1. **裸名即捕获，点号才是值**——`case MAX` 绑定、`case Color.RED` 比较
2. **序列看长度，映射看包含**——序列模式要求逐位对齐，映射模式只查指定键
3. **通配放最后，具体放前面**——匹配自上而下，第一个命中即生效
4. **守卫放业务，模式放结构**——结构判断交模式，数值/字符串条件交守卫
5. **类模式要位置，先备 match_args**——手写类要声明，dataclass 免操心
6. **字符串不是序列**——str/bytes 不解构，字面量或守卫来处理

### 8.3 自测练习（先自己写，再看参考实现）

**T1（映射 + 守卫）**：写函数 `validate_config(cfg)`，要求：`{"mode": "prod", "retries": n}` 中 `n` 在 1~5 之间返回 `"合法生产配置"`；`mode` 为 `"dev"` 时任意 retries 返回 `"开发配置"`；其余返回 `"非法配置"`。

**T2（序列 + OR）**：写函数 `parse_http(line)` 解析 `"GET /users"`、`"POST /users"`、`"DELETE /users/42"` 三种请求行，返回 `(method, path, param_or_None)`。

**T3（类模式 + 递归）**：定义 dataclass `Node(value, children)`（children 是 `list[Node]`），写递归函数 `tree_sum` 计算所有 value 之和；叶子（children 为空）直接返回，未知结构返回 0。

**T4（陷阱排查）**：下面的函数想区分 `0` 和其他整数，为什么永远输出“其他”？改正确它。

```python
def probe(x):
    match x:
        case n:
            return f"其他: {n}"
        case 0:
            return "零"
```

In [46]:
# 8-4 参考实现与自测验证

# T1 参考实现：映射 + 守卫
def validate_config(cfg):
    match cfg:
        case {"mode": "prod", "retries": int(n)} if 1 <= n <= 5:   # ① 映射 + int() 类子模式 + 守卫
            return "合法生产配置"
        case {"mode": "dev"}:                                      # ② 开发模式只看键
            return "开发配置"
        case _:
            return "非法配置"

assert validate_config({"mode": "prod", "retries": 3}) == "合法生产配置"   # ③ 边界内
assert validate_config({"mode": "prod", "retries": 9}) == "非法配置"      # ④ 超边界
assert validate_config({"mode": "dev", "retries": 99}) == "开发配置"      # ⑤ dev 不限
print("T1 全部通过")

# T2 参考实现：序列 + OR + 守卫
def parse_http(line):
    match line.split():                       # ⑥ 按 token 序列匹配
        case ["GET", "/users"]:               # ⑦ 精确两段
            return ("GET", "/users", None)
        case ["POST", "/users"]:
            return ("POST", "/users", None)
        case ["DELETE", path] if path.startswith("/users/"):       # ⑧ 守卫做前缀判断，分支体内提取参数
            return ("DELETE", "/users", path.removeprefix("/users/"))
        case [method, path]:                  # ⑨ 通用兜底：任何 method path
            return (method, path, None)
        case _:
            return (None, None, None)

assert parse_http("GET /users") == ("GET", "/users", None)
assert parse_http("POST /users") == ("POST", "/users", None)
assert parse_http("DELETE /users/42") == ("DELETE", "/users", "42")
print("T2 全部通过")

# T3 参考实现：dataclass 类模式 + 递归
from dataclasses import dataclass, field

@dataclass
class Node:
    value: int
    children: list = field(default_factory=list)   # ⑩ 默认空列表，避免可变默认值陷阱

def tree_sum(node):
    match node:
        case Node(value=v, children=[]):           # ⑪ 叶子：children 恰为空列表（序列字面量模式）
            return v
        case Node(value=v, children=kids):         # ⑫ 内部节点：递归求和
            return v + sum(tree_sum(k) for k in kids)
        case _:                                     # ⑬ 防御性兜底
            return 0

tree = Node(1, [Node(2, []), Node(3, [Node(4, [])])])   # ⑭ 1 + 2 + (3 + 4) = 10
assert tree_sum(tree) == 10
assert tree_sum(Node(7, [])) == 7
print("T3 全部通过")

# T4 答案：case n 是捕获模式，匹配一切，case 0 成为死分支。
# 修正：把具体模式（0）放在捕获之前，或直接用守卫。
def probe_fixed(x):
    match x:
        case 0:                 # ⑮ 具体模式在前
            return "零"
        case n:                 # ⑯ 捕获兜底在后
            return f"其他: {n}"

assert probe_fixed(0) == "零"
assert probe_fixed(5) == "其他: 5"
print("T4 全部通过（原因：裸名字 n 是捕获模式，会吞噬后续所有分支）")

T1 全部通过
T2 全部通过
T3 全部通过
T4 全部通过（原因：裸名字 n 是捕获模式，会吞噬后续所有分支）


### 8.4 结语

> **模式匹配不是简单的 switch-case，而是「检查结构 + 解构数据 + 绑定变量」的三合一工具；它最适合处理嵌套、异构、结构化的数据。**

选型一句话：**简单值比较用 if-elif，结构化解构与类型分发用 match，高频纯查表用字典**。